<a href="https://colab.research.google.com/github/bandi137/Asteroids/blob/main/Asteroids.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
import requests
import pandas as pd
from datetime import datetime
import plotly.express as px

# 1. API lekérés beállítása
# Használhatod a saját NASA API kulcsodat, de a 'DEMO_KEY' is működik korlátozottan
API_KEY = 'DEMO_KEY'
today = datetime.now().strftime('%Y-%m-%d')
url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={today}&end_date={today}&api_key={API_KEY}"

print(f"Adatok lekérése a mai napra ({today})...")
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    # Az objektumok listájának kinyerése a mai dátum alól
    asteroids = data['near_earth_objects'][today]

    # 2. Adatok feldolgozása listába
    rows = []
    for neo in asteroids:
        name = neo['name']
        # Becsült átmérő (átlagérték méterben)
        dia_min = neo['estimated_diameter']['meters']['estimated_diameter_min']
        dia_max = neo['estimated_diameter']['meters']['estimated_diameter_max']
        avg_diameter = (dia_min + dia_max) / 2

        # Relatív sebesség és elvétési távolság
        details = neo['close_approach_data'][0]
        speed = float(details['relative_velocity']['kilometers_per_hour'])
        miss_distance = float(details['miss_distance']['kilometers'])

        rows.append({
            'Név': name,
            'Átmérő (m)': round(avg_diameter, 2),
            'Sebesség (km/h)': round(speed, 2),
            'Távolság (km)': round(miss_distance, 2)
        })

    # 3. DataFrame létrehozása és megjelenítése
    df = pd.DataFrame(rows)
    print("\nMai földközeli objektumok táblázata:")
    display(df)

    # 4. Interaktív Plotly grafikon készítése
    fig = px.scatter(
        df,
        x="Távolság (km)",
        y="Sebesség (km/h)",
        size="Átmérő (m)",
        color="Átmérő (m)",
        hover_name="Név",
        title=f"Földközeli aszteroidák helyzete ({today})",
        labels={"Távolság (km)": "Távolság a Földtől (km)", "Sebesség (km/h)": "Sebesség (km/h)"},
        template="plotly_dark",
        color_continuous_scale=px.colors.sequential.Viridis,
        size_max=60
    )

    # Grafikon stílusának finomhangolása
    fig.update_layout(
        title_font_size=22,
        xaxis_showgrid=True,
        yaxis_showgrid=True
    )

    fig.show()

else:
    print(f"Hiba történt az API lekérés során: {response.status_code}")

Adatok lekérése a mai napra (2026-11-23)...

Mai földközeli objektumok táblázata:


,Név,Átmérő (m),Sebesség (km/h),Távolság (km)
0,(2019 SB6),18.95,21375.31,31460548.42
